# Advanced Python — Iterators, Generators, Function Copying, Closures & Decorators

## Theory, worked examples and advanced practice workbook

This notebook focuses deeply on five connected Python topics:

1. Iterators and the iterator protocol
2. Generators, `yield`, `yield from` and lazy evaluation
3. Function objects, references, copying and closure capture
4. Closures and higher-order functions
5. Decorators, decorator factories, stacking and metadata preservation

It also practises related skills such as `*args`, `**kwargs`, `functools.wraps`, `partial`, `itertools`, exceptions, type hints and testing.

## How to study this notebook

- Read each theory section and run its short example.
- Predict the output before executing the code.
- Modify the example and observe what changes.
- Complete the five advanced practice problems at the end.
- Each problem has its own Markdown cell followed immediately by a blank code cell.
- Practice sections contain requirements and test scenarios, not solutions.

## Learning outcomes

By the end, you should be able to explain how Python obtains values from iterables, build lazy data pipelines, reason about function references and closure state, and write reliable decorators that preserve the wrapped function's behaviour.

# 1. Functions are objects in Python

In Python, a function is an object. It can be assigned to a variable, stored in a list or dictionary, passed to another function, and returned from a function.

This is the foundation of higher-order functions, closures and decorators.

## 1.1 Assigning a function to another variable

A function name refers to a function object. Assigning it to another variable creates another reference to the same function; it does not create a new function.

In [ ]:

def calculate_total(price, quantity):
    return price * quantity

get_total = calculate_total

print(calculate_total(250, 3))
print(get_total(250, 3))
print(calculate_total is get_total)


## 1.2 Passing a function as an argument

A function can receive another function as an argument. This is useful when an operation should be configurable without rewriting the caller.

In [ ]:

def apply_operation(numbers, operation):
    results = []
    for number in numbers:
        results.append(operation(number))
    return results

def square(value):
    return value ** 2

def add_tax(value):
    return value * 1.18

values = [100, 200, 300]

print(apply_operation(values, square))
print(apply_operation(values, add_tax))


## 1.3 Returning a function

A function can return another function. The returned function can retain access to variables from the outer function. This leads to closures, covered later.

In [ ]:

def create_greeting(message):
    def greet(name):
        return f"{message}, {name}!"
    return greet

say_hello = create_greeting("Hello")
say_welcome = create_greeting("Welcome")

print(say_hello("Aman"))
print(say_welcome("Riya"))


## 1.4 Function references, aliases and copying

These operations are different:

- `alias = original` creates another reference to the same function object.
- `copy.copy(original)` generally returns the same function object; it does not clone the function's code and independent execution environment.
- `copy.deepcopy(original)` also does not create an independent function implementation.
- `functools.partial()` creates a new callable that pre-fills some arguments. It is useful for configuring a callable, but it is not a deep copy of the function.
- Creating a genuinely new function object requires deliberately constructing one, for example with `types.FunctionType`; this is an advanced operation and does not automatically clone closure state or referenced globals.

Do not confuse copying a function reference with copying the data a function uses.

In [ ]:

import copy
from functools import partial

def calculate_discount(price, rate):
    return price - (price * rate)

alias = calculate_discount
shallow = copy.copy(calculate_discount)
deep = copy.deepcopy(calculate_discount)
student_discount = partial(calculate_discount, rate=0.10)

print(alias is calculate_discount)
print(shallow is calculate_discount)
print(deep is calculate_discount)
print(student_discount(1000))


# 2. Iterables, iterators and the iterator protocol

## 2.1 Iterable

An iterable is an object from which an iterator can be obtained. Lists, tuples, strings, dictionaries, sets and files are common examples.

An object is iterable when it can provide an iterator, usually through `__iter__()`.

## 2.2 Iterator

An iterator produces values one at a time. It implements:

- `__iter__()` — returns the iterator itself.
- `__next__()` — returns the next item or raises `StopIteration` when exhausted.

An iterator normally keeps state, so it remembers where it is in the sequence.

## 2.3 Example: list versus iterator

In [ ]:

numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))
print(next(iterator))
print(next(iterator))

# Calling next(iterator) again raises StopIteration.


## 2.4 Building a custom iterator

A custom iterator is useful when values must be generated according to your own state or rules.

The example below creates an iterator that produces a range of invoice numbers.

In [ ]:

class InvoiceNumberIterator:
    def __init__(self, start, stop):
        self.current = start
        self.stop = stop

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration

        invoice_number = f"INV-{self.current:04d}"
        self.current += 1
        return invoice_number


invoices = InvoiceNumberIterator(1, 4)

for invoice in invoices:
    print(invoice)


## 2.5 Iterable versus iterator

| Iterable | Iterator |
|---|---|
| Can provide an iterator | Produces the next value |
| Often reusable | Usually consumed as it advances |
| Example: list | Example: `iter(list)` |
| `iter(obj)` obtains an iterator | `next(obj)` obtains the next value |

A list can create a fresh iterator more than once. A generator object, by contrast, is a single-use iterator: once exhausted, it does not restart automatically.

# 3. Generators and lazy evaluation

A generator is a convenient way to create an iterator. A generator function uses `yield` to produce a value and pause its execution. When the next value is requested, execution resumes from that point.

## 3.1 Generator function

In [ ]:

def generate_invoice_numbers(start, stop):
    current = start

    while current < stop:
        yield f"INV-{current:04d}"
        current += 1


invoice_stream = generate_invoice_numbers(1, 4)

print(next(invoice_stream))
print(next(invoice_stream))

for invoice in invoice_stream:
    print(invoice)


## 3.2 `return` versus `yield`

- `return` finishes a function and sends back its final result.
- `yield` produces a value while preserving the generator's execution state.
- A generator can yield many values over time.
- A generator function does not execute its body fully when called; it returns a generator object.

## 3.3 Generator expression

A generator expression looks similar to a list comprehension, but it produces values lazily.

In [ ]:

numbers = range(1, 11)

squares_list = [number ** 2 for number in numbers]
squares_generator = (number ** 2 for number in numbers)

print(squares_list)
print(next(squares_generator))
print(next(squares_generator))


## 3.4 Lazy evaluation and memory

A list comprehension builds the complete list immediately. A generator expression produces each item only when requested.

Use generators when:
- Processing a large file.
- Reading records in batches.
- Transforming a long or potentially unbounded stream.
- Passing values through a pipeline one at a time.

A generator is not automatically faster in every situation. It is especially useful when avoiding unnecessary storage matters.

## 3.5 `yield from`

`yield from` delegates iteration to another iterable or generator.

In [ ]:

def generate_department_reports():
    yield "HR report"
    yield "Finance report"

def generate_all_reports():
    yield "Management report"
    yield from generate_department_reports()
    yield "Operations report"

for report in generate_all_reports():
    print(report)


## 3.6 Generator state and exhaustion

A generator preserves local variables and its execution position between `next()` calls. Once it raises `StopIteration`, it is exhausted.

If you need to iterate over the same data again, create a new generator by calling the generator function again, or use a reusable iterable that can produce fresh iterators.

In [ ]:

def count_up_to(limit):
    for value in range(1, limit + 1):
        yield value

stream = count_up_to(3)

print(list(stream))
print(list(stream))  # Empty: the generator has already been consumed.
print(list(count_up_to(3)))  # A new generator starts again.


# 4. Closures and variable capture

A closure is an inner function that remembers variables from its enclosing scope even after the outer function has finished executing.

Closures are useful for:
- Configurable functions.
- Maintaining small pieces of private state.
- Creating decorator factories.
- Building reusable validation or filtering rules.

## 4.1 Basic closure

In [ ]:

def create_multiplier(factor):
    def multiply(number):
        return number * factor
    return multiply

double = create_multiplier(2)
triple = create_multiplier(3)

print(double(10))
print(triple(10))


## 4.2 `nonlocal` and updating closure state

An inner function can read a variable from its enclosing function. To reassign that enclosing variable, use `nonlocal`.

In [ ]:

def create_counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

counter_a = create_counter()
counter_b = create_counter()

print(counter_a())
print(counter_a())
print(counter_b())


## 4.3 Late binding in closures

A common closure bug occurs when inner functions capture a variable that changes later, such as a loop variable. The inner function looks up that variable when it is called.

A common solution is to capture the current value through a default argument or create a separate enclosing scope for each function.

In [ ]:

operations = []

for multiplier in range(1, 4):
    operations.append(lambda value, factor=multiplier: value * factor)

print([operation(10) for operation in operations])


## 4.4 Closure versus a class

A closure is often suitable for a small function with a little retained state. A class may be clearer when state has several fields, multiple operations or a longer lifecycle.

Do not force every stateful operation into a closure.

# 5. Decorators

A decorator is a callable that takes a function (or another callable), adds or changes behaviour, and returns a callable.

Decorators are commonly used for:
- Logging
- Timing
- Authentication and access checks
- Caching
- Validation
- Retry policies

## 5.1 Basic decorator

In [ ]:

def log_call(func):
    def wrapper():
        print("Function execution started")
        result = func()
        print("Function execution completed")
        return result
    return wrapper

@log_call
def generate_report():
    print("Generating report...")
    return "Report ready"

print(generate_report())


## 5.2 Decorators with arbitrary arguments

A general-purpose decorator should support different function signatures. Use `*args` and `**kwargs` in the wrapper and forward them to the original function.

In [ ]:

def log_call(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        return result
    return wrapper

@log_call
def calculate_invoice(price, quantity, tax_rate=0.18):
    subtotal = price * quantity
    return subtotal + (subtotal * tax_rate)

print(calculate_invoice(500, 2, tax_rate=0.05))


## 5.3 Preserve metadata with `functools.wraps`

Without `wraps`, the decorated function's visible name and docstring are usually those of the wrapper. `functools.wraps` copies important metadata and sets the wrapper's `__wrapped__` reference.

In [ ]:

from functools import wraps

def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Executing: {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_call
def create_invoice(customer_id):
    """Create an invoice for a customer."""
    return f"Invoice created for {customer_id}"

print(create_invoice.__name__)
print(create_invoice.__doc__)
print(create_invoice(101))


## 5.4 Decorator factory

A decorator factory accepts configuration and returns a decorator. It usually uses a closure to retain the configuration.

In [ ]:

from functools import wraps

def repeat_execution(times):
    if times < 1:
        raise ValueError("times must be at least 1")

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat_execution(3)
def send_notification(user):
    print(f"Notification sent to {user}")
    return True

print(send_notification("Aman"))


## 5.5 Stacking decorators

When decorators are stacked, the decorator closest to the function is applied first. At call time, the outer wrapper runs before it delegates to the inner wrapper.

In [ ]:

from functools import wraps

def first(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("First: before")
        result = func(*args, **kwargs)
        print("First: after")
        return result
    return wrapper

def second(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Second: before")
        result = func(*args, **kwargs)
        print("Second: after")
        return result
    return wrapper

@first
@second
def process_payment():
    print("Payment processed")

process_payment()


## 5.6 Important decorator design rules

1. Use `*args` and `**kwargs` when the decorator should support arbitrary signatures.
2. Return the wrapped function's result.
3. Use `functools.wraps`.
4. Decide explicitly how exceptions should behave. Most transparent decorators should let the original exception propagate.
5. Avoid mutable state shared accidentally between decorated functions.
6. Be careful when retrying functions that perform non-idempotent actions.
7. Test functions with no arguments, positional arguments, keyword arguments and mixed arguments.

# 6. Practice Problems

The following five problems are intentionally more challenging than syntax exercises. Each requires you to combine multiple concepts and implement realistic behaviour.

**Instructions:** Do not write all logic in one function. Use small functions or classes where appropriate, add type hints, handle edge cases, and test your implementation.

# Practice 1 — Custom Pagination Iterator

<badge>Advanced</badge>

## Scenario

You are developing a data access layer for an analytics application. A dataset may contain millions of records, so the application must expose records page by page instead of returning the entire dataset at once.

Build a reusable `Paginator` that behaves as an iterator.

## Requirements

1. Create a `Paginator` class that accepts an iterable source and a positive page size.
2. Implement `__iter__()` and `__next__()`.
3. Each call to `next()` should return the next page as a list.
4. The final page may contain fewer records than the configured page size.
5. Once all records have been returned, raise `StopIteration`.
6. Reject page sizes less than one.
7. Track the current page number.
8. Support an optional maximum number of pages.
9. Do not assume that the source is a list; it may be a generator or another iterator.
10. Document whether the paginator can be restarted and why.

## Required test scenarios

- Source contains 10 records with page size 3.
- Source contains exactly 6 records with page size 3.
- Source is empty.
- Source is a generator expression.
- Page size is zero or negative.
- Maximum page count stops iteration before the source is exhausted.
- Calling `next()` after exhaustion raises `StopIteration`.

## Advanced extension

Add a `map_pages(transform)` generator that lazily transforms each page without loading all pages into memory.

## Expected outcome

A reusable iterator that can paginate lists and one-pass data streams while maintaining correct state.

# Practice 2 — Lazy Log Processing Pipeline

<badge>Advanced</badge>

## Scenario

A production application writes large log files containing millions of lines. You need to build a processing pipeline that reads logs lazily, parses valid entries, filters events and produces a summary without loading the entire file into memory.

## Sample log format

```text
2026-10-01 10:15:30|INFO|User login successful
2026-10-01 10:16:05|ERROR|Database connection failed
2026-10-01 10:17:10|WARNING|High memory usage
2026-10-01 10:18:00|INFO|Report generated
```

## Requirements

1. Create a generator that reads a file line by line.
2. Parse each line into a structured record containing timestamp, level and message.
3. Reject malformed records according to a documented policy.
4. Create a generator that filters by one or more log levels.
5. Create a generator that filters records by a time range.
6. Use `yield from` to combine multiple log sources.
7. Create a summary containing counts by log level.
8. Support a configurable batch size for processing.
9. Ensure the pipeline can process a generator input.
10. Do not convert the complete log stream into a list merely to filter or count it.

## Required test scenarios

- Process a file with valid INFO, ERROR and WARNING entries.
- Handle a malformed line.
- Filter only ERROR records.
- Filter records between two timestamps.
- Process two files as one stream.
- Process an empty file.
- Process a large generated stream and verify that records are produced lazily.

## Advanced extension

Create a generator that yields only the first N matching records and stops reading once that limit is reached.

## Expected outcome

A composable lazy-processing pipeline suitable for large text files and multiple data sources.

# Practice 3 — Configurable Function Factory and Closure-Based State

<badge>Advanced</badge>

## Scenario

A subscription platform needs reusable pricing functions for different customer categories. Each pricing function should retain its own discount configuration and usage statistics without relying on global variables.

Build this using closures and higher-order functions.

## Requirements

1. Implement `create_pricing_rule(discount_rate, category_name)`.
2. The returned function should accept a base price and calculate the discounted price.
3. Validate that the discount rate is within the allowed range from 0 to 1.
4. Each returned pricing function must retain its own call count.
5. Use `nonlocal` to update the call count.
6. Expose a separate function or controlled mechanism to retrieve the call count.
7. Create separate rules for student, premium and standard customers.
8. Ensure updating one rule's state does not affect another rule.
9. Use `Decimal` for money calculations.
10. Document what state is captured by each closure.

## Required test scenarios

- Create three independent pricing rules.
- Verify each rule applies its own discount.
- Call one rule several times and verify its counter.
- Verify other rules' counters remain unchanged.
- Reject negative discount rates and rates above one.
- Handle zero and negative base prices according to a documented policy.
- Verify repeated calls preserve the expected state.

## Advanced extension

Allow a pricing rule to be updated through a separate controlled function while keeping the closure's internal state private.

## Expected outcome

A set of independent configurable functions that retain private state and demonstrate closure behaviour clearly.

# Practice 4 — Production-Ready Decorator Toolkit

<badge>Advanced</badge>

## Scenario

A software team wants a small reusable decorator library for application functions. The library must provide execution logging, timing, retry handling and argument validation.

The decorators must work with different function signatures and must not accidentally change successful return values.

## Required decorators

### A. `@log_execution`
- Log the wrapped function name.
- Log safe argument information.
- Log success or failure.
- Re-raise exceptions after recording failure.
- Mask sensitive keyword arguments such as password, token and api_key.

### B. `@measure_time`
- Measure duration with `time.perf_counter()`.
- Record duration for both successful and failed calls.
- Return the original result unchanged.

### C. `@retry(max_attempts, exceptions, delay)`
- Implement as a decorator factory.
- Retry only configured exception types.
- Do not retry unrelated exceptions.
- Raise the final exception when attempts are exhausted.
- Keep retry count and delay configurable.

### D. `@validate_positive`
- Validate configured argument names.
- Reject zero or negative values for those arguments.
- Work with positional and keyword invocation where practical.
- Do not call the wrapped function if validation fails.

## General requirements

1. Use `functools.wraps` in every decorator.
2. Use `*args` and `**kwargs` to forward arguments.
3. Preserve the original return value.
4. Keep decorator configuration in closures.
5. Support decorator stacking.
6. Define and document exception behaviour.
7. Do not log secrets.
8. Avoid retrying operations that may create duplicate side effects unless the caller explicitly chooses to do so.

## Required test scenarios

- Decorate a function with no arguments.
- Decorate a function with positional and keyword arguments.
- Verify function name and docstring are preserved.
- Verify a successful return value is unchanged.
- Verify a failed call is logged and the exception propagates.
- Verify a transient error is retried.
- Verify a permanent error is not retried.
- Verify the maximum attempt count.
- Verify a negative amount is rejected before the wrapped function runs.
- Stack all four decorators and verify their combined behaviour.

## Advanced extension

Support `async def` functions without blocking the event loop. Keep synchronous and asynchronous wrappers correct and test them separately.

## Expected outcome

A reusable decorator package with clean argument forwarding, metadata preservation, predictable exceptions and automated tests.

# Practice 5 — Streaming Data Pipeline with Decorators and Closures

<badge color="discovery">Capstone · Advanced</badge>

## Scenario

An analytics platform receives a continuous stream of sales events. It must validate, transform, filter and aggregate the events while keeping memory usage low. The pipeline should also measure each processing stage and allow configuration through closures and decorators.

## Event structure

Each event contains:
- `event_id`
- `customer_id`
- `category`
- `amount`
- `timestamp`

## Requirements

### Event source
1. Implement a generator that produces events one at a time.
2. Implement a custom iterator that reads events in configurable batches.
3. Reject invalid batch sizes.
4. Ensure the source can be used without converting it to a list.

### Processing stages
5. Create a generator that validates incoming events.
6. Create a generator that filters events by category.
7. Create a generator that transforms amounts using a configurable closure.
8. Use `yield from` to compose at least two event sources.
9. Use a generator expression for one simple transformation or filter.
10. Keep processing lazy until aggregation or export requests values.

### Decorator integration
11. Use a timing decorator on the aggregation operation.
12. Use a logging decorator on the export operation.
13. Use a configurable validation decorator or closure for event rules.
14. Preserve function metadata and return values.

### Aggregation and export
15. Calculate total sales by category.
16. Calculate average transaction value.
17. Identify the highest-value event.
18. Export processed records to CSV incrementally.
19. Track invalid, skipped and successfully processed event counts.
20. Handle empty input without crashing.

## Required test scenarios

- Process a small list of valid events.
- Process a generator source.
- Combine two sources.
- Filter one category.
- Reject negative amounts and missing IDs.
- Verify the transformation closure uses its configured rate.
- Verify aggregation results for known sample values.
- Export a large stream without creating a full intermediate list.
- Verify timing and logging decorators preserve results.
- Process an empty stream.

## Architecture constraints

Keep these responsibilities separate:

- Event generation
- Validation
- Filtering and transformation
- Aggregation
- Export
- Monitoring

Do not put the complete pipeline inside one large class or function.

## Advanced extensions

- Add checkpointing so a long-running pipeline can resume.
- Add duplicate event detection.
- Add configurable retry for transient export failures.
- Add a reusable pipeline builder that accepts processing functions.
- Compare memory use between eager list processing and lazy generator processing.

## Expected outcome

A small, extensible streaming framework that combines iterators, generators, closures and decorators in one realistic application.

# Final Revision Checklist

## Iterators
- [ ] I can explain iterable versus iterator.
- [ ] I can use `iter()` and `next()`.
- [ ] I can implement `__iter__()` and `__next__()`.
- [ ] I understand `StopIteration`.
- [ ] I know why many iterators are single-use.

## Generators
- [ ] I can write a generator function using `yield`.
- [ ] I can explain generator execution and suspension.
- [ ] I can use generator expressions.
- [ ] I can use `yield from`.
- [ ] I can explain lazy evaluation and its memory benefits.
- [ ] I know when a generator is unsuitable, such as when repeated traversal is required.

## Function objects and copying
- [ ] I understand that functions are first-class objects.
- [ ] I can pass a function as an argument and return one from another function.
- [ ] I can distinguish an alias from a new callable.
- [ ] I understand why `copy.copy()` and `copy.deepcopy()` do not clone a function's execution environment.
- [ ] I can use `functools.partial()` to pre-fill arguments.

## Closures
- [ ] I can explain enclosing scope and captured variables.
- [ ] I can create a closure.
- [ ] I can use `nonlocal` to update enclosing state.
- [ ] I understand late binding in loops.
- [ ] I can choose between a closure and a class.

## Decorators
- [ ] I can write a basic decorator.
- [ ] I can support arbitrary function arguments using `*args` and `**kwargs`.
- [ ] I use `functools.wraps`.
- [ ] I can create a decorator factory.
- [ ] I understand closure-based decorator configuration.
- [ ] I can stack decorators and explain the order.
- [ ] I preserve return values and exception behaviour.

## Completion standard

You are ready to move on when you can complete all five practice problems without copying a solution, explain the design choices, and write tests for normal and failure cases.

**Suggested order:** Practice 1 → Practice 2 → Practice 3 → Practice 4 → Practice 5. The fifth problem is the integration project; use it to demonstrate that you can combine the concepts rather than merely define them.